# Part I: Initial Benchmark
We are going to benchmark our current implementation, first we want to set $n$ and $m$ so that they can be on a reasonably sized instance. 

We use the ks_func function we implemented in HW2, and we set $n$  and $m$  to be 1,000,000 instead of 1000 to make the memory bottleneck and performance issues more observable.  And we use the benchmark tool to see the execute time and see what'll happen.



In [1]:
using Plots
using Revise
includet("../src/ks-stat.jl")

In [2]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(123)
n0 = randn(1000000)
m0 = randn(1000000)
@btime ks_func(n0,m0)
@benchmark ks_func(n0,m0)


  237.963 ms (175367 allocations: 101.31 MiB)


BenchmarkTools.Trial: 20 samples with 1 evaluation per sample.
 Range (min … max):  240.065 ms … 273.987 ms  ┊ GC (min … max): 5.93% … 9.04%
 Time  (median):     256.410 ms               ┊ GC (median):    6.16%
 Time  (mean ± σ):   256.945 ms ±  10.077 ms  ┊ GC (mean ± σ):  6.17% ± 3.19%

  ▁ ▁          ▁█ ▁▁   ▁ ▁▁         ▁▁ ▁ ▁    █   ▁   ▁       █  
  █▁█▁▁▁▁▁▁▁▁▁▁██▁██▁▁▁█▁██▁▁▁▁▁▁▁▁▁██▁█▁█▁▁▁▁█▁▁▁█▁▁▁█▁▁▁▁▁▁▁█ ▁
  240 ms           Histogram: frequency by time          274 ms <

 Memory estimate: 98.21 MiB, allocs estimate: 58822.

The current implementation takes approximately 237 ms to run. More importantly, it allocates 101 MiB of memory. This indicates a significant inefficiency, likely due to the creation of intermediate data structures during the labeling and sorting process.

# Part II: Iterative Code Optimization 
## - Round 1
## 1. Identify Bottlenecks

In [4]:
ks_func(n0[1:10], m0[1:10])

Profile.clear()

ProfileCanvas.@profview ks_func(n0, m0)

## Function and Line Number:
Based on the result from the Flame Graph, the primary bottleneck is located in the `ks_func` function, specifically at line 103 in ks-stat.jl. This line corresponds to the `sort!` operation on the combined array.

## Nature of the Bottleneck:
The bottleneck is primarily caused by unnecessary memory allocation, which leads to excessive computation:
1. Memory Allocation caused by string, since the code creates a new tuple containing string (value,"label") for every single data point for a single run, it causes a huge volume of pressure on system's memory and the Collector.
2. The Flame Graph shows that 67% of the execution time is spent in `sort!`. We might need to change the `sort!` a bit.

# 2. Alternative Implementations:

### Rationale for Alternatives
The bottleneck in the original code was the creation of intermediate `(value, label)` tuples and sorting a mixed array.
* **Alternative 1 (Separate Sorting + Two-Pointer):** Instead of mixing the arrays, we sort `X` and `Y` separately. We use a "Two-Pointer" algorithm to calculate the KS statistic in faster way which will avoid more allocations.

In [5]:
# Implementation for the alternatives two-pointer function 


function ks_func_2pt(X::AbstractVector,Y::AbstractVector)

    max_diff= 0.0
    cdf_x = 0.0
    cdf_y = 0.0
    n = length(X)
    m = length(Y)
    sort_x = sort(X)
    sort_y =sort(Y)
    i=1
    j=1

    while i<=n || j<=m 
        if i>n
            current_val = sort_y[j]
        elseif j>m
            current_val = sort_x[i]
        else 
            current_val = min(sort_x[i],sort_y[j])
        end
    

        while i <=n && sort_x[i]== current_val #??current_val_x == sort_x[i]
            cdf_x +=1/n
            i+=1
        end

        while j <=m && sort_y[j]== current_val
            cdf_y +=1/m
            j+=1
        end

        curr_diff = abs(cdf_x-cdf_y)
        if curr_diff > max_diff
            max_diff =curr_diff
        end

    end

    return max_diff

end        

ks_func_2pt (generic function with 1 method)

# 3. Benchmark Alternatives:

In [6]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(123)
n1 = randn(1000000)
m1 = randn(1000000)
@btime ks_func_2pt(n1,m1)
@benchmark ks_func_2pt(n1,m1)

  89.257 ms (33639 allocations: 30.38 MiB)


BenchmarkTools.Trial: 49 samples with 1 evaluation per sample.
 Range (min … max):   89.757 ms … 158.775 ms  ┊ GC (min … max): 0.00% … 35.63%
 Time  (median):      95.965 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   102.143 ms ±  13.723 ms  ┊ GC (mean ± σ):  6.10% ±  9.93%

     ▃▅█                                                         
  ▄▄████▁▄█▅█▁▁▁▁▁▁▁▁▁▁▄▁▁▅▄▁▅▁▁▁▁▁▁▄▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄ ▁
  89.8 ms          Histogram: frequency by time          159 ms <

 Memory estimate: 30.38 MiB, allocs estimate: 33639.

In [8]:
ks_func_2pt(n1[1:10], m1[1:10])

Profile.clear()

ProfileCanvas.@profview  ks_func_2pt(n1, m1)

As we can see the alternative implementation significantly improved the bottleneck:

Execution Time: The median runtime dropped from 237 ms to 89 ms, achieving a ~2x speedup.

Memory Efficiency: Total memory allocation was reduced by approximately 70% (from 101 MiB to 30.39 MiB).

Conclusion: The Flame Graph previously identified `sort!` as the main culprit. By switching to a Two-Pointer approach with separate sorting of raw Float64 vectors, we bypassed the expensive process of sorting tuples with strings.

# 4. Update Code Base:

Therefore we have improved the bottleneck, we update our code base and add the `ks_func_2pt` function inside to better visualize the difference. We re-run the unit tests to verify that nothing is broken.


In [9]:
# check for correctness

n = randn(1000)
m = randn(2000)

# run benchmarks
v1 = @btime ks_func(n,m)
v2 = @btime ks_func_2pt(n,m)

@assert isapprox(v1, v2)

  164.000 μs (13 allocations: 140.93 KiB)
  74.000 μs (25 allocations: 56.88 KiB)


We can see that the two results are approximately equal, we can say we are correct. And we update our `ks-stat,jl` in `src` and `ks-unit-test.jl` in `test`.

By unit test, we can see that all the unit tests pass, nothing is broken , then our result is correct for the new optimization version. 

We commit and push our new code to GitHub.

### As we can see from the above results, the behavior of the KS_function has been improved. But we still want more.

# Part II: Iterative Code Optimization
## Round 2
## 1.Identify Bottlenecks: 


Function and Line Number: The bottleneck is identified in the `sort!` function, specifically where the input arrays $X$ and $Y$ are sorted at the beginning of the ks_func. 

In the flame graph from Round 1, this operation now occupies the vast majority of the total execution time.

Nature of the Bottleneck: The nature of this bottleneck is Computationally Intensive Serial Execution. The remaining cost is the mathematical necessity of sorting two large arrays.

The Key Issue: 
The current implementation sorts array $X$ and array $Y$ sequentially. Since the sorting of $X$ is mathematically independent of $Y$, running them serially on a single CPU core is inefficient. Since we have multiple cores in our computer, we can consider using more cores, which can reduce the total time from sum of both sorting operations ($T_x + T_y$) to the maximum of the two ($\max(T_x, T_y)$).

In [10]:
Random.seed!(2026)
n = randn(1000)
m = randn(1000)
@btime ks_func_2pt(n,m)
@benchmark ks_func_2pt(n,m)

  45.700 μs (23 allocations: 35.27 KiB)


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  50.500 μs …  27.002 ms  ┊ GC (min … max): 0.00% … 99.52%
 Time  (median):     70.800 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   86.705 μs ± 273.509 μs  ┊ GC (mean ± σ):  3.10% ±  1.00%

  ▃▄█▄▃▂▂                                                       
  ████████▇▆▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  50.5 μs         Histogram: frequency by time          269 μs <

 Memory estimate: 35.27 KiB, allocs estimate: 23.

## 2. Alternative Implementations:
Rationale for the alternative:
Since the sorting of array $X$ and array $Y$ are mathematically independent, they can be performed simultaneously on separate CPU cores.

Alternative (Parallel Sorting): We utilize Julia's multi-threading capabilities (Base.Threads). By spawning a new thread to sort $Y$ while the main thread sorts $X$, we can reduce the sorting wall-clock time by up to 50% hopefully, as the operations run in parallel rather than sequentially.

In [13]:
using Base.Threads
println("Current Number of Threads: ", Threads.nthreads())

Current Number of Threads: 1


In [14]:
function ks_func_parallel(X::AbstractVector, Y::AbstractVector)
    # We spawn a task to sort Y on a separate thread
    y_task = Threads.@spawn sort(Y)
    sx =sort(X)
    sy = fetch(y_task)::Vector{eltype(Y)} #Wait for Y to finish the sorting and fetch the result
    i =1
    j =1
    cdf_x =0.0
    cdf_y =0.0
    n = length(sx)
    m = length(sy)
    max_diff= 0.0

    while i<=n || j<=m 
        if i>n
            current_val = sy[j]
        elseif j>m
            current_val = sx[i]
        else 
            current_val = min(sx[i],sy[j])
        end
    

        while i <=n && sx[i]== current_val #??current_val_x == sort_x[i]
            cdf_x +=1/n
            i+=1
        end

        while j <=m && sy[j]== current_val
            cdf_y +=1/m
            j+=1
        end

        curr_diff = abs(cdf_x-cdf_y)
        if curr_diff > max_diff
            max_diff =curr_diff
        end

    end

    return max_diff
end

ks_func_parallel (generic function with 1 method)

# 3. Benchmark Alternatives

In [17]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas


Random.seed!(123)
# Increase data size to 1 million to observe parallel benefits
n2 = randn(1000000)
m2 = randn(1000000)

println("Two Pointer Method: ")
@btime ks_func_2pt($n2, $m2)
println("Parallel Sorting: ")
@btime ks_func_parallel($n2, $m2)
@benchmark ks_func_parallel(n2,m2)

Two Pointer Method: 
  89.776 ms (26 allocations: 28.70 MiB)
Parallel Sorting: 
  63.922 ms (31 allocations: 28.70 MiB)


BenchmarkTools.Trial: 45 samples with 1 evaluation per sample.
 Range (min … max):   84.105 ms … 180.827 ms  ┊ GC (min … max): 0.00% … 23.67%
 Time  (median):     108.884 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   113.672 ms ±  21.202 ms  ┊ GC (mean ± σ):  4.77% ±  9.96%

      ▁ ▄▄ ▁  ▁█▁  ▁█  ▁▁                                        
  ▆▁▁▆█▁██▆█▆▁███▆▁██▆▆██▆▁▁▆▆▁▁▁▆▆▁▁▁▁▁▁▁▁▁▆▁▆▁▁▆▁▁▆▁▁▁▁▁▁▁▁▁▆ ▁
  84.1 ms          Histogram: frequency by time          181 ms <

 Memory estimate: 28.70 MiB, allocs estimate: 32.

In [18]:
ks_func_parallel(n2[1:10], m2[1:10])

Profile.clear()

ProfileCanvas.@profview  ks_func_parallel(n2, m2)

As we can see from the result, he parallel implementation significantly improved the bottleneck.

#### Round 1: The median runtime was 87 ms.

#### Round 2: Parallel Execution : The median runtime dropped to 61 ms.

Analysis of the Bottleneck:
Before round 2, the CPU had to sort array $X$ and then sort array $Y$ sequentially. The total sorting time was roughly $T_{sortX} + T_{sortY}$. After using the parallel methods, we allowed array $Y$ to be sorted on a separate core at the same time as array $X$. This effectively reduced the sorting wall-clock time. The execution time has been reduced.
Additionally, the total memory usage remained identical at 28.70 MB, this means our parallelization introduced negligible memory overhead  while the executive speed goes up.

# 4. Update Code Base:
We check for correctness and use re-run your unit tests to verify that nothing is broken.



In [19]:
# check for correctness

n = randn(1000)
m = randn(2000)

# run benchmarks
v1 = @btime ks_func(n,m)
v2 = @btime ks_func_2pt(n,m)
v3 = @btime ks_func_parallel(n,m)

@assert isapprox(v1, v2) && isapprox(v1,v3) 

#We re-run our unit tests to verify that nothing is broken

  170.000 μs (13 allocations: 140.92 KiB)
  80.200 μs (25 allocations: 56.88 KiB)
  78.700 μs (30 allocations: 57.19 KiB)


We can see that the three results are approximately equal, we can say we are correct. And we add the new function `ks_func_parallel` , we update our `ks-stat,jl` in `src` and `ks-unit-test.jl` in `test`. We want to compared the three results so I don't remove the original version of the KS functions.

We see that all the unit tests pass, nothing is broken , then our result is correct for the new optimization version. 

We commit and push our new code to GitHub.

# Part III: Final Benchmark
We use the same test instance from Part I to benchmark the performance of our code.

1. Benchmark your final implementation. Describe the performance gains both in
terms of execution time and memory.


In [ ]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(123)
# Increase data size to 1 million to observe parallel benefits
n3 = randn(1000000)
m3 = randn(1000000)

println("The Original Method: ")
@btime ks_func($n3, $m3)
println("Two Pointer Method: ")
@btime ks_func_2pt($n3, $m3)
println("Parallel Sorting: ")
@btime ks_func_parallel($n3, $m3)

@benchmark ks_func_parallel(n3,m3)

The Original Method: 
  246.391 ms (12 allocations: 91.55 MiB)
Two Pointer Method: 


In [ ]:
ks_func_parallel(n[1:10], m[1:10])

Profile.clear()

ProfileCanvas.@profview  ks_func_parallel(n3, m3)

## Detailed Analysis:

### 1.Execution Time:

The final parallel implementation is nearly 4 times faster than the original code. This dramatic improvement stems from two distinct optimization phases:

Algorithmic Efficiency (Round 1): By switching to the "Two-Pointer" method, we eliminated the overhead of creating thousands of tuples and the costly array concatenation (vcat). This alone reduced the time from 269ms to 89ms.

Parallel Computing (Round 2): By identifying sort! as the remaining bottleneck and applying Threads.@spawn, we utilized multi-core processing to sort the two large arrays simultaneously. This further reduced the runtime from 89ms to 61ms.

### 2. Memory Efficiency Gains
We achieved a massive around 69% reduction in memory usage (dropping from 91.55 MiB to 28.70 MiB).
